# 航班价格监控 · EDA 00 — 数据总览与质量审计

**说明**：本 notebook 及同目录 01/02 均对数据库 **只读**（sqlite `mode=ro`），不会修改、删除 `flight_monitor.db` 里的任何数据；派生结果只落在内存 / 本文件夹的图。

数据来源表：
- `flight_prices`：每次爬取，对“某航向 × 某航班号 × 某起飞日”记录一条**价格快照**（字段含当时价格、机型、起降机场/时刻等）
- `price_alerts`：触发变价提醒的事件（87 条）
- `monitor_log`：每次监控运行日志（166 条）

我们会先确认：**爬了多少、覆盖哪些航向/航班/起飞日、采样是否连续（断档审计）、字段是否完整、每条轨迹（航班×起飞日）被采样多深**。

## 0. 准备

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "eda"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import common as C

# 中文字体与配色（dataviz 参考色板顺序）
mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
})
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
sns.set_palette(PALETTE)

df = C.load_flight_prices()
df = C.add_features(df)
print("flight_prices 行数:", len(df), "| 列:", list(df.columns))

## 1. 数据规模与时间范围
一次“爬取批次” = 同一 `crawl_time` 下写入的一组快照。全库共多少次爬取？覆盖了哪些起飞日？

In [ ]:
print("监控批次数(不同 crawl_time):", df["crawl_time"].nunique())
print("抓取时间范围:", df["crawl_time"].min(), "→", df["crawl_time"].max())
print("覆盖起飞日范围:", df["flight_date"].min(), "→", df["flight_date"].max(),
      f"（共 {df['flight_date'].nunique()} 个起飞日）")
print("监控航班号数:", df["flight_no"].nunique())
print("价格范围: %d ~ %d，均值 %.0f" % (df["price"].min(), df["price"].max(), df["price"].mean()))

1.结论：全库 **59,966 条快照 / 149 个爬取批次**，抓取期 2026-06-30 → 09-03，覆盖 **95 个起飞日**（至 10/02）、**45 个航班号**；价格 200–5,230 元、均值 645 元。数据量足够做“按 lead 对齐”的时间维分析；注意 5,230 等极值来自全价舱口径（见 4b）。

## 2. 航向 / 航班 / 航司构成
按“航向（方向）”看：各自有多少快照、多少航班号、多少航司、覆盖多少起飞日。

In [ ]:
route = (df.groupby("route_label")
           .agg(rows=("price", "size"),
                flights=("flight_no", "nunique"),
                airlines=("airline", "nunique"),
                dep_dates=("flight_date", "nunique"))
           .reset_index())
route["rows%"] = (100 * route["rows"] / route["rows"].sum()).round(1)
route

In [ ]:
# 北京侧机场（出发方向 bjs→·；到达方向 ·→bjs）分布——用于后续“机场定价差”
pd.crosstab(df["route_label"], df["bj_airport"])

2.结论：北京⇄泉州是绝对主体——去程 29,654 条（49.5%）+ 回程 25,708 条（42.9%），各 9 个航班号、5 家航司、覆盖全部 95 个起飞日；北京→厦门仅 4,604 条（7.7%）、27 个航班但只有 8 个起飞日（后加入的 alert_only 航线）。北京侧机场分布：大兴（去程 22,348 / 回程 19,375）明显多于首都（7,306 / 6,333），即这条线以大兴为主场。

## 3. 爬取时间线 & 断档审计
每天有几个爬取批次、几条快照？有没有**整天没跑/没写入**的空档（关系到“日级时间序列”是否连续）？

In [ ]:
daily = (df.groupby(df["crawl_time"].str[:10])
           .agg(batches=("crawl_time", "nunique"), rows=("price", "size")))
daily.index = pd.to_datetime(daily.index)
full = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
missing = [d.date() for d in full if d not in daily.index]
print("观察期天数:", len(full), "| 有数据的天数:", len(daily), "| 整天缺档:", len(missing))
print("缺档日期:", missing if missing else "（无）")

fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(daily.index, daily["batches"], width=0.8, color="#2a78d6")
for d in missing:
    ax.axvspan(pd.Timestamp(d) - pd.Timedelta(hours=12),
               pd.Timestamp(d) + pd.Timedelta(hours=12), color="#e34948", alpha=0.12)
ax.set_title("每天爬取批次数量（红底=整天缺档）")
ax.set_xlabel("日期"); ax.set_ylabel("批次")
fig.tight_layout(); plt.show()

3.结论：66 天观察期中 60 天有数据，**6 个整天缺档**（7/27、7/28 连续两天，及 7/30、8/3、8/8、8/13），红底位置与“笔记本休眠/关机”时间吻合。日常批次数多为 1–3 次/天（配置 180 分钟间隔，理论最多 8 次），8 月中旬后批次更稀、单日批次波动大（图右半段）——**日级序列不连续、日内分辨率有限**；轨迹级（同一航班×起飞日）分析不受影响，但按天聚合的“趋势”读数要打折扣。

## 4. 字段完整性
关键文本字段是否有空值/空串（直接影响按航司、机型、机场做聚合）。

In [ ]:
text_cols = ["route_from_name", "route_to_name", "airline", "aircraft_type",
             "departure_airport", "arrival_airport", "departure_time", "arrival_time"]
rows = []
for c in text_cols:
    bad = int(df[c].isna().sum() + (df[c].astype(str) == "").sum())
    rows.append({"字段": c, "空值/空串": bad, "占比%": round(100 * bad / len(df), 3)})
pd.DataFrame(rows)

4.结论：8 个文本字段**零缺失/零空串**，按航司、机型、机场、时刻做聚合可以放心使用；`price` 为整数最低可订价，未见负数或异常量级（极值见 4b）。

## 4b. 异常价格与口径审计

`price` 是“抓取时该航班最低可订价”，但少数快照明显是**全价舱/临近售罄后的口径跳变**，
与普通折扣调整不是一回事——若不区分，会夸大“临期涨价”的幅度统计与价格上限：

- 全库最大价 **5230 只有 1 条**（CZ8973，7/1 起飞当天抓到）；
- 7/5 当天 **多个国航/山航厦门航班同价 3090**（6 个航班号）；9/2 批次 **3 个中联航班同价 3460** —— 典型的“该日只剩全价舱”；
- 价格 ≥2000 的快照有 **59% 落在 lead≤3**（见下方 (a)）。

另一种规模大得多的“堆集效应”：**500/520 两个价占了全部快照约 13%**（中联航固定 520、深航/厦航/河北 500 带），
价格分布呈明显双峰——这是航司平台价结构，不是异常值。

**特别警告——北京→厦门的“同价共振”**：同一航向×起飞日×批次里 ≥5 个航班完全同价的情形共 **96 组、涉及 522 行**，
几乎全部来自北京→厦门（如 9/9–9/13 连续两周 600/660、7/5 的 450/750/790），且该航线单航班同批次内的
价格档数明显少于主线（中位 1–2 vs 主线 7–13，见下方 (c)）——**疑似抓到的不是逐航班最低价，
而是页面级“¥xxx 起”占位价**，建议对照抓取页面验证；本系列其余分析（01/02）都以北京⇄泉州为主线，
厦门数据应谨慎使用。


In [ ]:
# (a) 高值快照集中在临期？—— price>=2000 的 lead 分布
hi = df[(df["price"] >= 2000) & (df["lead_days"] >= 0)]
print("价格≥2000 的临期快照数:", len(hi),
      "| lead≤3 占比: %.0f%%" % (100 * (hi["lead_days"] <= 3).mean()))
print(hi["lead_days"].value_counts().sort_index().head(6).to_string())

# (b) 极值样本
print("\nTop 8 高价快照:")
print(df.nlargest(8, "price")[["flight_no", "flight_date", "crawl_time",
                               "lead_days", "price"]].to_string(index=False))

# (c) 日级“同价共振”：同(航向,起飞日,批次)内 >=5 个航班完全同价
g = (df.groupby(["route_label", "flight_date", "crawl_time", "price"])["flight_no"]
       .nunique().rename("nf").reset_index())
plat = g[g["nf"] >= 5].sort_values("nf", ascending=False)
print("\n日级同价共振(>=5航班同日同批):", len(plat), "组, 航班次合计", int(plat["nf"].sum()))
print("按航向分布:")
print(plat["route_label"].value_counts().to_string())
print("\nnf 最大的 8 组:")
print(plat.head(8).to_string(index=False))

# 每航班单批次内价格档数：厦门 vs 主线
per = (df.groupby(["flight_no", "crawl_time"])["price"].nunique().rename("nlevel")
         .reset_index().groupby("flight_no")["nlevel"].median().rename("中位档数").reset_index())
per["route"] = per["flight_no"].map(df.groupby("flight_no")["route_label"].first())
print("\n每航班在同批次内的价格档数（跨批次取中位）:")
print(per.groupby("route")["中位档数"].agg(["median", "min", "max"]).round(1).to_string())

# (d) 500/520 价格堆集：占比 + 航司构成
mass = df[df["price"].isin([500, 520])]
print("\nprice∈{500,520}: %d 行 (%.1f%%)" % (len(mass), 100 * len(mass) / len(df)))
print(mass.groupby(["price", "airline"])
      .agg(n=("flight_no", "size"), nf=("flight_no", "nunique"))
      .sort_values("n", ascending=False).head(10).to_string())


4b.结论：**三条口径要点**——① 高值快照（≥2000 元，233 条）59% 落在 lead≤3，属“仅剩全价舱”的口径跳变（5230×1、3090×6、3460×3），计入“普通涨价”会显著夸大临期涨幅；② 价格呈平台双峰：500/520 共 8,049 行（13.4%），中联航固定 520、深航/厦航/河北 500 带，是航司平台价而非异常值；③ **北京→厦门有数据质量风险**：96 组“同日≥5 航班同价”共振中 95 组来自该航线，且单航班同批次价格档数中位仅 2（主线 8.5–9）——疑似抓到的是页面级“¥xxx 起”而非逐航班最低价，后续对厦门航线的价格结论应谨慎。

## 5. 轨迹采样深度（航班 × 起飞日）
一条“轨迹” = 某个航班在某个起飞日上，被反复爬取的价格序列。
采样越深，越能做“价格怎么随时间变”的分析。

In [ ]:
ts = C.trajectory_summary(df)
print("轨迹总数:", len(ts))
print(ts["n_samples"].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
ts["n_samples"].hist(bins=range(0, 70, 3), ax=ax, color="#1baf7a", edgecolor="white")
ax.set_title("每条轨迹被采样次数分布")
ax.set_xlabel("采样次数"); ax.set_ylabel("轨迹数")
fig.tight_layout(); plt.show()

# 轨迹里出现过几个不同价格（价格台阶数）
print("\n—— 轨迹内出现过的不同价格数分布 ——")
print(ts["n_price_levels"].value_counts().sort_index().to_string())

5.结论：共 **1,774 条轨迹**（航班×起飞日），采样次数中位 **34 次**（25%/75% 分位 24/46，最多 67 次）——绝大多数轨迹足够深，可以做“价格随时间演化”的轨迹级分析。轨迹内常见 **5–7 个不同价格档**（7 档最多、227 条），说明价格档位化、跳变式变化。少数采样 <10 次的轨迹集中在观察窗两端（7 月初新加入、10 月初新日期），读临期行为时注意这些薄样本。

## 6. 价格水平概览（按航向）
注意 `price` 是“抓取那一刻该航班的最低可订价”快照，不是舱位全价；
快照里混着早/晚、机型、不同航司，这里只看大致的价格带。

In [ ]:
df[df["lead_days"] >= 0].groupby("route_label")["price"].describe(
    percentiles=[.05, .25, .5, .75, .95]).round(0)

fig, ax = plt.subplots(figsize=(8, 3.5))
order = df["route_label"].value_counts().index.tolist()
sns.boxplot(data=df[df["lead_days"] >= 0], x="route_label", y="price", order=order,
            palette=PALETTE[:len(order)], width=.5, ax=ax)
ax.set_title("价格快照分布（按航向，仅起飞前/当天）")
fig.tight_layout(); plt.show()

6.结论：三条航向的快照价格带都集中在 **400–800 元**、中位约 600 元；500/520 平台价使下沿出现明显尖峰（对应 4b 的 13.4% 双峰），右侧长尾稀疏（1,500–5,230）且均为临期/全价口径。北京→泉州与泉州→北京的箱体形态几乎一致，北京→厦门整体略高（中位区 550–800）。

## 7. 变价事件与告警/日志一览
价格相邻两次“真的变了”算一次变价事件（见 `common.change_events`）。up/down 平衡吗？幅度多大？

In [ ]:
ev = C.change_events(df)
print("变价事件总数:", len(ev))
print(ev["direction"].value_counts().to_string())
print("\n变价幅度(绝对值，元): 中位数 %.0f / 均值 %.0f"
      % (ev["diff"].abs().median(), ev["diff"].abs().mean()))
print("\n|变化| 分布（元）:")
print(ev["diff"].abs().describe(percentiles=[.5, .75, .9, .99]).round(1).to_string())

# 告警表
alerts = C.load_price_alerts()
print("\nprice_alerts:", len(alerts), "行")
if len(alerts):
    print(alerts["change_percent"].describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())

logs = C.load_monitor_log()
print("\nmonitor_log:", len(logs), "行; status 分布:")
print(logs["status"].value_counts().to_string())

7.结论：全库共 **11,369 次变价事件**，涨（5,662）跌（5,707）基本平衡；单次幅度中位 **90 元**、均值 146 元（75% 在 170 元内），长尾最大 3,260 元——留意长尾正是 4b 的“全价跳变”贡献。告警表 87 条（变动% 中位 −2.3%、极值 +54.8%），监控日志 167 次全部 OK。监控系统本身运行稳定，事件/告警口径一致。

## 小结（质量结论）
- **数据量足以做时间维分析**：约 6 万条快照、149 批次，覆盖 6/30–9/3 的抓取与至 10/2 的起飞日。
- **主战场是北京⇄泉州**（约 5.5 万条）；北京→厦门是后加的、样本少（只覆盖个别起飞日）。
- **存在整天缺档**（见第 3 节红底），做“按天趋势”时要留意；轨迹级分析不受影响。
- **轨迹足够深**：大部分轨迹采样 ≥20 次，可做“价格随时间演化”和 lead-time 对齐。
- 字段基本无缺失；价格确为“该航班当时可订最低价”的口径（若要严格验证，需对照抓取页面对某一舱位快照）。
- **口径注意**：价格≥2000 的快照 59% 在 lead≤3（全价舱/临期售罄态），5230/3090/3460 均属这种口径跳变，不计入“普通涨价”；另有 500/520 平台价堆集（约 13%），价格分布呈明显双峰；北京→厦门存在大量日级同价共振，疑似页面级“起价”，其结论需谨慎（详见 4b）。

下一步（notebook 01）：把轨迹按 **距起飞天数（lead）** 对齐，回答“什么时候买便宜、现在买划不划算、去程回程差异”。

